Clinical QA Grounding System

Preporcessing

In [1]:
# Import necessary libraries and mount your drive, upload necessary files according to the requirement
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [2]:
# Import necessary libraries and load admissions dataset
import pandas as pd
admissions = pd.read_csv('/content/drive/My Drive/ADMISSIONS.csv')

In [3]:
# Load diagnosis dataset
diagnosis = pd.read_csv('/content/drive/My Drive/Diagnosis.csv', sep='\t', encoding='UTF-16', usecols=["ROW_ID", "SUBJECT_ID", "HADM_ID", "SEQ_NUM", "ICD9CODE"])

In [4]:
# Load Procedure's dataset
procedures = pd.read_csv("/content/drive/My Drive/Procedures.csv", sep="\t", encoding="UTF-16",low_memory=False)

In [5]:
# Load Notevents Dataset
events = pd.read_csv("/content/drive/My Drive/NOTEEVENTS.csv",low_memory=False)

In [6]:
#Load Patients and prescriptions dataset
patients = pd.read_csv("/content/drive/My Drive/PATIENTS.csv")
prescriptions = pd.read_csv("/content/drive/My Drive/PRESCRIPTIONS.csv",low_memory=False)

Preprocessing

In [7]:
# Import necessary libraries
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
import spacy

# Drop rows with missing values in crucial fields
events = events.dropna(subset=['TEXT'])
diagnosis = diagnosis.dropna(subset=['ICD9CODE'])
procedures = procedures.dropna(subset=['ICD9CODE'])
prescriptions = prescriptions.dropna(subset=['DRUG'])
admissions = admissions.dropna(subset=['HADM_ID'])

In [8]:
# Drop the columns which are not required for the task to avoid confusion
events = events.drop(columns=['STORETIME', 'CGID', 'CHARTTIME'])

In [9]:
# Drop the null values
events = events.dropna(subset=['HADM_ID'])

In [10]:
events["ISERROR"] = events["ISERROR"].fillna(0)

In [11]:
# Extract Admission Date & Discharge Date correctly
events["ADMISSION_DATE"] = events["TEXT"].str.extract(r'Admission Date:\s*\[\*\*(\d{4}-\d{1,2}-\d{1,2})\*\*\]')
events["DISCHARGE_DATE"] = events["TEXT"].str.extract(r'Discharge Date:\s*\[\*\*(\d{4}-\d{1,2}-\d{1,2})\*\*\]')

In [12]:
# Change the column name to merge them
procedures.rename(columns={
    "SUBJECTID": "SUBJECT_ID",
    "HADMID": "HADM_ID"
}, inplace=True)

In [13]:
# Ensure matching dtypes for merge keys
events["SUBJECT_ID"] = events["SUBJECT_ID"].astype(str)
events["HADM_ID"] = events["HADM_ID"].astype(str)
admissions["SUBJECT_ID"] = admissions["SUBJECT_ID"].astype(str)
admissions["HADM_ID"] = admissions["HADM_ID"].astype(str)
patients["SUBJECT_ID"] = patients["SUBJECT_ID"].astype(str)
diagnosis["SUBJECT_ID"] = diagnosis["SUBJECT_ID"].astype(str)
diagnosis["HADM_ID"] = diagnosis["HADM_ID"].astype(str)
procedures["SUBJECT_ID"] = procedures["SUBJECT_ID"].astype(str)
procedures["HADM_ID"] = procedures["HADM_ID"].astype(str)
prescriptions["SUBJECT_ID"] = prescriptions["SUBJECT_ID"].astype(str)
prescriptions["HADM_ID"] = prescriptions["HADM_ID"].astype(str)

RAG QA System

In [ ]:
#install everytime when your runtine disconnects.
!pip install faiss-cpu

In [ ]:
#install everytime when your runtine disconnects.
!pip install openai --upgrade
!pip install nltk pandas numpy faiss-cpu sentence-transformers

In [ ]:
# Set OpenAI API key securely
import os
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print('API key loaded from Colab secrets.')
except Exception:
    import getpass
    os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter your OpenAI API key: ')
    print('API key set via prompt.')


In [ ]:
#install everytime when your runtine disconnects.
!pip install gradio

Set up the RAG System

In [ ]:
import os
import openai
from openai import OpenAI
from transformers import T5Tokenizer, T5ForConditionalGeneration
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize

# Setup
nltk.download('punkt_tab')
openai_client = OpenAI()

# Load models
retrieval_model = SentenceTransformer('all-MiniLM-L6-v2')
tokenizer_t5 = T5Tokenizer.from_pretrained('t5-small')
model_t5 = T5ForConditionalGeneration.from_pretrained('t5-small')

# Load and process clinical notes
def load_clinical_notes(file_path, max_notes=10000):
    data = pd.read_csv(file_path, low_memory=False)
    clinical_notes = data['TEXT'].dropna().tolist()
    return clinical_notes[:max_notes]

def encode_sentences(sentences, batch_size=32):
    return retrieval_model.encode(sentences, convert_to_numpy=True, batch_size=batch_size, show_progress_bar=True)

def build_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings.astype(np.float32))
    return index

def retrieve_relevant_sentences(query, all_sentences, faiss_index, top_k=8):
    q_embed = retrieval_model.encode([query], convert_to_numpy=True)
    _, indices = faiss_index.search(q_embed.astype(np.float32), top_k)
    return [all_sentences[i] for i in indices[0]]

# Generate answer using OpenAI API with citations
def generate_openai_answer(clinician_q, patient_q, top_sentences):
    top_sentences = [s.strip() for s in top_sentences if len(s.strip()) > 10]
    context_block = "\n".join([f"{i+1}: {sent}" for i, sent in enumerate(top_sentences)])

    user_prompt = (
        f"You are a clinical assistant tasked with answering a medical question. "
        f"Given the sentences from clinical notes, generate a professional answer that clearly explains the reason for the treatment or event. "
        f"Cite evidence from the notes using numbered sentence references like (1), (2). Limit your answer to 75 words.\n\n"
        f"Context:\n{context_block}\n\n"
        f"Patient Question: {patient_q}\n"
        f"Clinician Question: {clinician_q}\n\n"
        f"Answer:"
    )

    response = openai_client.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": "You are a medical assistant who generates concise, professional explanations with citations from clinical notes."},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.3,
        max_tokens=200
    )

    answer = response.choices[0].message.content.strip()

    # Truncate to 75 words
    words = answer.split()
    if len(words) > 75:
        answer = " ".join(words[:75]) + "..."

    return answer

# Main function to retrieve the notes from the clinical note dataset and generate a stuctured answer
def interactive_qa_with_openai_rag(file_path, max_notes=1000, max_sentences=5000):
    clinical_notes = load_clinical_notes(file_path, max_notes)

    print("Splitting sentences...")
    all_sentences = []
    for note in clinical_notes:
        sentences = sent_tokenize(note.strip())
        all_sentences.extend(sentences)
        if len(all_sentences) >= max_sentences:
            break
    all_sentences = all_sentences[:max_sentences]
    print(f"Total sentences: {len(all_sentences)}")

    embeddings = encode_sentences(all_sentences)
    index = build_faiss_index(embeddings)

    patient_q = input("Enter the patient question:\n").strip()
    clinician_q = input("Enter the clinician question:\n").strip()

    combined_query = patient_q + " " + clinician_q
    top_sentences = retrieve_relevant_sentences(combined_query, all_sentences, index, top_k=8)

    print("\nTop Retrieved Sentences:")
    for i, sent in enumerate(top_sentences, 1):
        print(f"({i}): {sent}")

    answer = generate_openai_answer(clinician_q, patient_q, top_sentences)

    print("\nGenerated Answer with Citations (Max 75 Words):")
    print(answer)
    return answer, top_sentences, clinician_q, patient_q

# Run the RAG QA pipeline
file_path = '/content/drive/MyDrive/NOTEEVENTS.csv'
answer, top_sentences, clinician_q, patient_q = interactive_qa_with_openai_rag(file_path, max_notes=10000, max_sentences=20000)

Gradio Interface

In [ ]:
import gradio as gr

# === New preprocessing before Gradio ===
# Prepare all_sentences and index before defining the Gradio app
print("Preparing sentences and FAISS index for Gradio app...")

# Load clinical notes
clinical_notes = load_clinical_notes(file_path, max_notes=10000)

# Split into sentences
all_sentences = []
for note in clinical_notes:
    sentences = sent_tokenize(note.strip())
    all_sentences.extend(sentences)
all_sentences = all_sentences[:20000]  # Limit to 20,000 sentences

# Encode and build FAISS index
embeddings = encode_sentences(all_sentences)
index = build_faiss_index(embeddings)

print(f"Prepared {len(all_sentences)} sentences and FAISS index.")

# === Gradio-compatible function ===
def interactive_qa_with_openai_rag_ui(patient_q, clinician_q):
    combined_query = patient_q + " " + clinician_q
    top_sentences = retrieve_relevant_sentences(combined_query, all_sentences, index, top_k=8)
    answer = generate_openai_answer(clinician_q, patient_q, top_sentences)

    retrieved_sentences_str = "\n".join([f"({i+1}) {s}" for i, s in enumerate(top_sentences)])
    return retrieved_sentences_str, answer

# === Build the Gradio app ===
demo = gr.Interface(
    fn=interactive_qa_with_openai_rag_ui,
    inputs=[
        gr.Textbox(label="Patient Question"),
        gr.Textbox(label="Clinician Question")
    ],
    outputs=[
        gr.Textbox(label="Top Retrieved Sentences"),
        gr.Textbox(label="Generated Answer with Citations")
    ],
    title="Clinical QA Assistant (RAG + GPT-4)",
    description="Enter a patient question and a clinician question. Retrieves relevant notes and generates a professional answer with citations."
)

# === Launch Gradio ===
demo.launch()

Evaluation

In [ ]:
#install everytime when your runtine disconnects.
# BERTScore
!pip install bert-score

# ROUGE scorer
!pip install rouge-score

Evaluate the answers

In [ ]:
# Import necessary libraries
from google.colab import drive
drive.mount('/content/drive')
import json, os

# Download test.final.json from the repo if not already present
if not os.path.exists('/content/test.final.json'):
    !wget -q -O /content/test.final.json https://raw.githubusercontent.com/sriya19/Clinical-Question-Answering-Model-using-MIMIC-III-LLMs/main/test.final.json
    print('Downloaded test.final.json')
else:
    print('test.final.json already present')

qa_id = 12  # Update this to your actual QA pair ID

# Load test.final.json
with open('/content/test.final.json') as f:
    test_data = json.load(f)

print(f'Loaded {len(test_data["data"])} articles from test.final.json')


In [ ]:
# Import libraries
import torch, os, json
from bert_score import score as bert_score
from rouge_score import rouge_scorer
from sentence_transformers import util

# Ensure test.final.json is downloaded
if not os.path.exists('/content/test.final.json'):
    import subprocess
    subprocess.run(['wget', '-q', '-O', '/content/test.final.json',
                    'https://raw.githubusercontent.com/sriya19/Clinical-Question-Answering-Model-using-MIMIC-III-LLMs/main/test.final.json'])

# Load test.final.json
with open('/content/test.final.json') as f:
    test_data = json.load(f)

# Flatten test.final.json into list of QA pairs
def flatten_qa_pairs(test_data):
    all_qas = []
    for article in test_data['data']:
        for para in article['paragraphs']:
            for qa in para['qas']:
                all_qas.append({
                    'id': qa['id'],
                    'question': qa['question'],
                    'answer': qa['answers'][0]['text']
                })
    return all_qas

all_gold_qas = flatten_qa_pairs(test_data)

# Find best matching QA
def find_most_similar_qa(all_gold_qas, user_question, top_k=1):
    questions = [qa['question'] for qa in all_gold_qas]
    embeddings_gold = retrieval_model.encode(questions, convert_to_tensor=True)
    embedding_user = retrieval_model.encode(user_question, convert_to_tensor=True)
    similarities = util.pytorch_cos_sim(embedding_user, embeddings_gold)[0]
    top_result = torch.topk(similarities, k=top_k)
    idx = top_result.indices[0].item()
    return all_gold_qas[idx]

# Evaluate Factuality
def evaluate_factuality(generated_answer, gold_answer):
    P, R, F1 = bert_score([generated_answer], [gold_answer], lang='en', verbose=False)
    return {
        'BERTScore Precision': P.item(),
        'BERTScore Recall': R.item(),
        'BERTScore F1': F1.item()
    }

# Evaluate Relevance
def evaluate_relevance(generated_answer, top_sentences, labels=None):
    if not labels:
        labels = [1] * len(top_sentences)
    essential = [s for i, s in enumerate(top_sentences) if labels[i] == 2]
    if not essential:
        essential = [s for i, s in enumerate(top_sentences) if labels[i] == 1]
    reference = ' '.join(essential)
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    rouge = scorer.score(reference, generated_answer)
    P, R, F1 = bert_score([generated_answer], [reference], lang='en', verbose=False)
    return {
        'ROUGE-1 F1': rouge['rouge1'].fmeasure,
        'ROUGE-L F1': rouge['rougeL'].fmeasure,
        'BERTScore F1': F1.item()
    }

# Combine both patient and clinician questions for matching
combined_user_q = patient_q + ' ' + clinician_q
matched_gold = find_most_similar_qa(all_gold_qas, combined_user_q)
gold_answer = matched_gold['answer']

print('\nMatched QA ID:', matched_gold['id'])

# Run evaluations
factuality_scores = evaluate_factuality(answer, gold_answer)
relevance_scores = evaluate_relevance(answer, top_sentences)

print('\n=== Factuality Evaluation ===')
for k, v in factuality_scores.items():
    print(f'{k}: {v:.4f}')

print('\n=== Relevance Evaluation ===')
for k, v in relevance_scores.items():
    print(f'{k}: {v:.4f}')

End of Notebook